# 🧩 RAG Lab: Different Ways to Chunk Podcast and PDF

**Goal:** Explore and compare chunking strategies for two content types:
- 🎙️ A Podcast audio file (we'll transcribe it first)
- 📄 A PDF document

**Strategies covered:**
1. Fixed-Size Chunking
2. Recursive Character Chunking
3. Token-Based Chunking
4. Semantic Chunking (advanced)
5. Visualization & Comparison
6. Final Recommendations

---
## 📦 Step 1: Install & Import Dependencies

📦 What each package does
🔗 langchain

Core framework for building LLM apps.

Used for:

- chains
- agents
- prompts
- RAG pipelines
- tool integration

Think: the orchestration layer.

🧱 langchain-community

Extra integrations maintained by the community.

Examples:
- document loaders
- vector databases
- embedding models
- external tools

Think: plug-ins for LangChain.

📄 pypdf

Reads and extracts text from PDFs.

Used when:
- loading documents for RAG
- parsing reports, ebooks, research papers

🤖 openai

Official OpenAI API client.

Used to:
- call GPT models
- generate embeddings
- chat completions

🔢 tiktoken

Tokenizer used by OpenAI models.

Helps with:
- counting tokens
- chunking documents correctly
- estimating cost

Very important in RAG.

🧠 sentence-transformers

Local embedding models (HuggingFace).

Used for:
- semantic search
- embedding text without API calls
- similarity matching

Example models:
- all-MiniLM-L6-v2
- mpnet-base-v2

📊 matplotlib

Plotting and visualization library.

Used to:
- visualize embeddings
- charts
- evaluation results

🔢 numpy
Core numerical computing library.

Used everywhere for:
- vectors
- matrices
- embeddings math

Many AI libraries depend on it.

🔐 python-dotenv

Loads environment variables from a .env file.

In [ ]:
# Install all required packages
!pip install langchain langchain-community pypdf openai tiktoken \
             sentence-transformers matplotlib numpy python-dotenv -q

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# LangChain splitters
from langchain.text_splitter import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)

# PDF loading
from pypdf import PdfReader

# Tokenization
import tiktoken

# OpenAI for transcription (Whisper API)
from openai import OpenAI

# Environment variables
from dotenv import load_dotenv
load_dotenv()

print('✅ All imports successful!')

---
## 🔑 Step 2: Configure API Key

In [ ]:
# Option A: Load from .env file (recommended)
# Create a .env file in this directory with: OPENAI_API_KEY=sk-...
load_dotenv()

# Option B: Set directly (not recommended for shared environments)
# os.environ['OPENAI_API_KEY'] = 'sk-your-key-here'

api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise ValueError('❌ OPENAI_API_KEY not found. Please set it in your .env file or environment.')

client = OpenAI(api_key=api_key)
print('✅ OpenAI client initialized')

## 🎙️ Step 3: Transcribe the Podcast Audio (Whisper API)

In this step we convert the podcast audio file into text using **OpenAI's Whisper API**,
which is a state-of-the-art speech recognition model.

### What this code does

**1. Configuration**  
We define two paths:
- `AUDIO_FILE_PATH` — the podcast audio file to transcribe (`.m4a`, `.mp3`, etc.)
- `TRANSCRIPT_FILE` — where to save the resulting text so we don't pay to re-transcribe on every run

**2. Smart caching**  
Before calling the API, the function checks if a transcript file already exists locally.
If it does, it loads it from disk instead of making an API call. This saves time and
API costs during development.

**3. Transcription via Whisper**  
If no saved transcript exists, the audio file is sent to OpenAI's `whisper-1` model
using `client.audio.transcriptions.create()`. We request `response_format='text'`
which returns a plain string (as opposed to JSON with timestamps).

**4. Save & reuse**  
The transcript is written to `podcast_transcript.txt` immediately after transcription,
so future runs skip the API call entirely.

**5. Stats preview**  
Finally we print basic statistics (character count, word count) and the first 500
characters so we can visually confirm the transcription looks correct.

> ⚠️ **Note:** The Whisper API has a 25MB file size limit. If your audio file exceeds
> this, see the extended version of this function in the troubleshooting section which
> automatically splits large files into chunks before transcribing.

In [ ]:
# ⚙️ CONFIGURE THIS: Path to your podcast audio file
AUDIO_FILE_PATH = 'The_Blueprint_For_Trustworthy_AI.m4a'  # Change to your actual file name
TRANSCRIPT_FILE = 'podcast_transcript.txt'  # Where to save the transcript

def transcribe_audio(audio_path: str, save_path: str) -> str:
    """
    Transcribe audio using OpenAI Whisper API.
    Saves the transcript to a local file to avoid re-transcribing.
    """
    # Check if we already have a saved transcript
    if Path(save_path).exists():
        print(f'📂 Loading existing transcript from {save_path}')
        with open(save_path, 'r', encoding='utf-8') as f:
            return f.read()
    
    print(f'🎙️ Transcribing {audio_path} with Whisper API...')
    print('   (This may take a minute depending on file size)')
    
    if not Path(audio_path).exists():
        raise FileNotFoundError(f'Audio file not found: {audio_path}')
    
    with open(audio_path, 'rb') as audio_file:
        response = client.audio.transcriptions.create(
            model='whisper-1',
            file=audio_file,
            response_format='text'  # Returns plain text
        )
    
    transcript = response  # response is a string when format='text'
    
    # Save for reuse
    with open(save_path, 'w', encoding='utf-8') as f:
        f.write(transcript)
    
    print(f'✅ Transcript saved to {save_path}')
    return transcript

# Run transcription
podcast_text = transcribe_audio(AUDIO_FILE_PATH, TRANSCRIPT_FILE)

print(f'\n📊 Podcast transcript stats:')
print(f'   Characters : {len(podcast_text):,}')
print(f'   Words      : {len(podcast_text.split()):,}')
print(f'\n📖 First 500 characters preview:')
print(podcast_text[:500])

## ⚠️ Issue 1: Whisper API File Size Limit (413 Error)

### What happened
When we tried to transcribe the podcast, the API returned a **413 error**:
```
APIStatusError: Error code: 413 - Maximum content size limit (26214400) exceeded
```

### Why it happened
The **OpenAI Whisper API has a hard 25MB file size limit** (26,214,400 bytes).  
Our podcast file `The_Blueprint_For_Trustworthy_AI.m4a` was **28.8MB** — just
slightly over the limit — so the API rejected the request entirely.

### How we solved it
We used the `pydub` library to **split the audio into smaller chunks** before
sending it to the API:

1. Load the full audio file into memory
2. Calculate how many chunks are needed to stay under 24MB each
3. Export each chunk as a temporary `.mp3` file
4. Transcribe each chunk separately via the Whisper API
5. Join all transcript pieces together with a space
6. Delete the temporary chunk files
7. Save the final combined transcript locally for reuse

This approach is transparent — the final transcript is identical to what
you would get if the file were small enough to send in one request.

> 💡 **Lesson learned:** Always check your audio file size before calling the
> Whisper API. For files close to the limit, use chunked transcription to avoid
> the error entirely.

In [ ]:
from pydub import AudioSegment

# Point pydub directly to your ffmpeg executables
AudioSegment.converter = r"C:\ffmpeg\fullbuild\bin\ffmpeg.exe"
AudioSegment.ffprobe   = r"C:\ffmpeg\fullbuild\bin\ffprobe.exe"

# Quick verification
import subprocess
result = subprocess.run([r"C:\ffmpeg\fullbuild\bin\ffmpeg.exe", "-version"], 
                       capture_output=True, text=True)
print(result.stdout[:200] if result.returncode == 0 else "❌ Still not working")

In [ ]:
import pydub.utils
import inspect

# See what command pydub builds when calling ffprobe
print("ffprobe path pydub uses:", pydub.utils.get_prober_name())
print("ffmpeg path pydub uses:", pydub.utils.get_encoder_name())

# Also check if our manual path actually works
import subprocess
result = subprocess.run(
    [r"C:\ffmpeg\fullbuild\bin\ffprobe.exe", "-version"],
    capture_output=True, text=True
)
print("\nffprobe direct call:")
print(result.stdout[:150] if result.returncode == 0 else f"❌ Error: {result.stderr[:150]}")

In [ ]:
import pydub.utils
from pydub import AudioSegment

# Override the path pydub uses internally
pydub.utils.get_prober_name = lambda: r"C:\ffmpeg\fullbuild\bin\ffprobe.exe"
pydub.utils.get_encoder_name = lambda: r"C:\ffmpeg\fullbuild\bin\ffmpeg.exe"

AudioSegment.converter = r"C:\ffmpeg\fullbuild\bin\ffmpeg.exe"
AudioSegment.ffprobe   = r"C:\ffmpeg\fullbuild\bin\ffprobe.exe"

# Verify pydub now sees the right paths
print("ffprobe path pydub uses:", pydub.utils.get_prober_name())
print("ffmpeg path pydub uses:", pydub.utils.get_encoder_name())
print("✅ pydub paths overridden successfully")

## 🔧 Issue 2 & 3: ffmpeg Not Found + Upgraded Transcription Function

### What happened
After solving the 413 error, we introduced `pydub` to split the audio file.
But a new error appeared — a `JSONDecodeError` deep inside pydub's internals:
```
JSONDecodeError: Expecting value: line 1 column 1 (char 0)
```

### Why it happened
`pydub` is just a Python wrapper — it relies on **ffmpeg**, a separate system
program, to actually read and process audio files. ffmpeg was installed on this
machine at `C:\ffmpeg\fullbuild\bin\` but was **not added to the system PATH**,
so pydub called `ffprobe` by name and got nothing back.

### How it was solved
Bypassed the PATH issue by **pointing pydub directly to the ffmpeg executables**:
```python
AudioSegment.converter = r"C:\ffmpeg\fullbuild\bin\ffmpeg.exe"
AudioSegment.ffprobe   = r"C:\ffmpeg\fullbuild\bin\ffprobe.exe"
pydub.utils.get_prober_name  = lambda: r"C:\ffmpeg\fullbuild\bin\ffprobe.exe"
pydub.utils.get_encoder_name = lambda: r"C:\ffmpeg\fullbuild\bin\ffmpeg.exe"
```

We also had to specify `format='m4a'` explicitly when loading the audio:
```python
audio = AudioSegment.from_file(audio_path, format='m4a')
```
Without this, pydub tries to detect the format automatically — which also
requires ffprobe to be working correctly.

### What this upgraded function does

| Feature | Description |
|---|---|
| **Caching** | Loads from disk if transcript already exists — no repeat API calls |
| **Size check** | Measures file size in MB before attempting transcription |
| **Auto-split** | If file > 24MB, splits into equal time-based chunks |
| **Chunk transcription** | Each chunk is transcribed separately and temp files deleted after |
| **Reassembly** | All chunk transcripts joined with a space into one final text |
| **Verification** | Prints file size, chunk count, and a 500-char preview on completion |

> 💡 **Lesson learned:** When using audio processing libraries on Windows,
> always verify that system dependencies like ffmpeg are accessible — either
> via PATH or by hardcoding the executable path as a fallback.

In [ ]:
from pydub import AudioSegment

# Quick verification
import subprocess
result = subprocess.run([r"C:\ffmpeg\fullbuild\bin\ffmpeg.exe", "-version"], 
                       capture_output=True, text=True)
print(result.stdout[:200] if result.returncode == 0 else " Still not working")

import math
from pydub import AudioSegment

AUDIO_FILE_PATH = 'The_Blueprint_For_Trustworthy_AI.m4a'   # ← your file
TRANSCRIPT_FILE = 'podcast_transcript.txt'

def transcribe_audio(audio_path: str, save_path: str) -> str:
    """
    Transcribe audio using OpenAI Whisper API.
    Automatically splits files larger than 24MB into chunks.
    Saves the transcript locally to avoid re-transcribing on reruns.
    """
    from pathlib import Path

    # If we already have a transcript saved, just load it
    if Path(save_path).exists():
        print(f'📂 Loading existing transcript from {save_path}')
        with open(save_path, 'r', encoding='utf-8') as f:
            return f.read()

    if not Path(audio_path).exists():
        raise FileNotFoundError(f'Audio file not found: {audio_path}')

    file_size_mb = Path(audio_path).stat().st_size / (1024 * 1024)
    print(f'📁 File size: {file_size_mb:.1f} MB')

    MAX_MB = 24  # Stay safely under the 25 MB API limit

    if file_size_mb <= MAX_MB:
        # Small enough — send directly
        print('🎙️ File is under 24MB, transcribing directly...')
        with open(audio_path, 'rb') as f:
            response = client.audio.transcriptions.create(
                model='whisper-1',
                file=f,
                response_format='text'
            )
        transcript = response

    else:
        # File too large — split into chunks and transcribe each
        print(f'  File is {file_size_mb:.1f} MB — splitting into chunks...')

        audio = AudioSegment.from_file(audio_path, format='m4a')
        total_ms = len(audio)

        # Calculate how many chunks we need
        num_chunks = math.ceil(file_size_mb / MAX_MB)
        chunk_ms = total_ms // num_chunks

        print(f'   Splitting into {num_chunks} chunks of ~{chunk_ms/60000:.1f} minutes each')

        transcripts = []

        for i in range(num_chunks):
            start_ms = i * chunk_ms
            end_ms = min((i + 1) * chunk_ms, total_ms)
            chunk = audio[start_ms:end_ms]

            chunk_path = f'_temp_chunk_{i}.mp3'
            chunk.export(chunk_path, format='mp3')

            chunk_size_mb = Path(chunk_path).stat().st_size / (1024 * 1024)
            print(f'   🎙️ Transcribing chunk {i+1}/{num_chunks} ({chunk_size_mb:.1f} MB)...')

            with open(chunk_path, 'rb') as f:
                response = client.audio.transcriptions.create(
                    model='whisper-1',
                    file=f,
                    response_format='text'
                )
            transcripts.append(response)

            # Clean up temp file
            Path(chunk_path).unlink()

        # Join all transcripts with a space
        transcript = ' '.join(transcripts)

    # Save for future reruns
    with open(save_path, 'w', encoding='utf-8') as f:
        f.write(transcript)

    print(f'✅ Transcript saved to {save_path}')
    return transcript

# Run it
podcast_text = transcribe_audio(AUDIO_FILE_PATH, TRANSCRIPT_FILE)

print(f'\n📊 Podcast transcript stats:')
print(f'   Characters : {len(podcast_text):,}')
print(f'   Words      : {len(podcast_text.split()):,}')
print(f'\n📖 First 500 characters preview:')
print(podcast_text[:500])

## 📄 Step 4: Load the PDF Document

### What this code does

In this step we extract all the text from the PDF file using **`pypdf`**,
a pure-Python library for reading PDF files.

### Why not just copy-paste the text?
A 73-page document would be impractical to handle manually. `pypdf` lets us
extract all text programmatically in seconds, preserving the reading order
of each page.

### How it works

**1. Configuration**  
We define `PDF_FILE_PATH` pointing to our local PDF file. Update this path
if your file has a different name or location.

**2. Page-by-page extraction**  
The function iterates through every page using `PdfReader` and calls
`page.extract_text()` on each one. Some pages may contain only images
(scanned content) with no extractable text — those are silently skipped
with the `if text:` check.

**3. Joining pages**  
Pages are joined with `'\n\n'` (double newline) between them. This is
important for downstream chunking — it gives `RecursiveCharacterTextSplitter`
a strong signal that a page boundary has occurred, so it can try to split
there before breaking mid-paragraph.

**4. Stats preview**  
We print character count, word count, and a 500-character preview to visually
confirm the extraction looks correct.



In [ ]:
# ⚙️ CONFIGURE THIS: Path to your PDF file
PDF_FILE_PATH = 'Living_Repository_AI_Literacy_Practices.pdf'  # Change to your actual file name

def load_pdf(pdf_path: str) -> str:
    """
    Extract all text from a PDF file using pypdf.
    Joins pages with double newlines to preserve structure.
    """
    if not Path(pdf_path).exists():
        raise FileNotFoundError(f'PDF file not found: {pdf_path}')
    
    reader = PdfReader(pdf_path)
    pages = []
    
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:  # Some pages may be images with no extractable text
            pages.append(text)
    
    full_text = '\n\n'.join(pages)  # Double newline between pages
    print(f'✅ Loaded PDF: {len(reader.pages)} pages')
    return full_text

# Load PDF
pdf_text = load_pdf(PDF_FILE_PATH)

print(f'\n📊 PDF text stats:')
print(f'   Characters : {len(pdf_text):,}')
print(f'   Words      : {len(pdf_text.split()):,}')
print(f'\n📖 First 500 characters preview:')
print(pdf_text[:500])

### Result from our document
- **Pages:** 73
- **Characters:** 226,399
- **Words:** 32,066

---
## 🔧 Step 5: Helper Functions

Before running any chunking strategy, we define three utility functions
that will be reused throughout the entire lab.

---

### 1. `count_tokens(text)` — Token Counter
Counts how many tokens a string contains using **tiktoken** with the
`cl100k_base` encoding — the same tokenizer used by GPT-4 and Claude.

This matters because LLMs don't process text character by character —
they process **tokens**. One token is roughly 4 characters in English,
but it varies: short common words like "the" are 1 token, while long or
unusual words can be 3-4 tokens. Measuring in tokens gives us a more
accurate picture of how much of an LLM's context window each chunk consumes.

---

### 2. `chunk_stats(chunks, label)` — Statistics Calculator
Takes a list of chunks and computes:

| Metric | Description |
|---|---|
| `count` | Total number of chunks produced |
| `avg_chars` | Average chunk size in characters |
| `char_range` | Min and max chunk size in characters |
| `avg_tokens` | Average chunk size in tokens |
| `token_range` | Min and max chunk size in tokens |

It prints a formatted summary and returns the stats as a dictionary
so we can store and compare results across strategies later.

---

### 3. `check_sentence_breaks(chunks)` — Quality Checker
Estimates what percentage of chunks end **mid-sentence** by checking
whether the last character of each chunk is terminal punctuation
(`.`, `!`, `?`, `"`, `'`).

If a chunk ends without terminal punctuation, it likely cut through
a sentence — which means the next chunk starts mid-thought, losing
context for the retrieval system.

> ⚠️ **Note:** This function was later improved to `check_sentence_breaks_v2()`
> after we discovered that `RecursiveCharacterTextSplitter` *consumes* the period
> when splitting on `'. '`, making clean cuts appear as mid-sentence breaks.
> The improved version detects true mid-word cuts instead.

In [ ]:
# Initialize tokenizer (matches GPT-4 / Claude token counting)
encoding = tiktoken.get_encoding('cl100k_base')

def count_tokens(text: str) -> int:
    """Count the number of tokens in a string."""
    return len(encoding.encode(text))

def chunk_stats(chunks: list, label: str) -> dict:
    """
    Compute statistics for a list of chunks.
    Returns a dict with count, char sizes, and token sizes.
    """
    char_lengths = [len(c) for c in chunks]
    token_lengths = [count_tokens(c) for c in chunks]
    
    stats = {
        'label': label,
        'count': len(chunks),
        'avg_chars': int(np.mean(char_lengths)),
        'min_chars': min(char_lengths),
        'max_chars': max(char_lengths),
        'avg_tokens': int(np.mean(token_lengths)),
        'min_tokens': min(token_lengths),
        'max_tokens': max(token_lengths),
        'char_lengths': char_lengths,
        'token_lengths': token_lengths
    }
    
    print(f'\n📊 {label}')
    print(f'   Chunks      : {stats["count"]}')
    print(f'   Avg chars   : {stats["avg_chars"]:,}')
    print(f'   Char range  : {stats["min_chars"]:,} – {stats["max_chars"]:,}')
    print(f'   Avg tokens  : {stats["avg_tokens"]}')
    print(f'   Token range : {stats["min_tokens"]} – {stats["max_tokens"]}')
    
    return stats

def check_sentence_breaks(chunks: list, sample_size: int = 10) -> float:
    """
    Estimate what percentage of chunks break mid-sentence.
    A chunk 'breaks' if it doesn't start with a capital letter
    (after stripping whitespace) or doesn't end with punctuation.
    """
    breaks = 0
    sample = chunks[:sample_size]
    
    for i, chunk in enumerate(sample[:-1]):  # Don't check last chunk
        # Check if chunk ends mid-sentence (no terminal punctuation)
        stripped = chunk.strip()
        if stripped and stripped[-1] not in '.!?"\'':
            breaks += 1
    
    pct = (breaks / len(sample)) * 100 if sample else 0
    return pct

print('✅ Helper functions ready')

---
## 📏 Step 6: Strategy 1 — Fixed-Size Chunking

### What is Fixed-Size Chunking?
Fixed-size chunking splits text every **N characters**, regardless of where
sentences, paragraphs, or ideas begin and end. It is the simplest possible
chunking strategy — a pure mechanical cut with no understanding of content.

### How it works in code
We use LangChain's `CharacterTextSplitter` with `separator=''`, which means
there is no preferred split point — it cuts at exactly N characters every time.
Two key parameters control the output:

| Parameter | Description |
|---|---|
| `chunk_size` | Maximum number of characters per chunk |
| `chunk_overlap` | How many characters to repeat at the start of the next chunk |

### Why experiment with different sizes?
Different chunk sizes produce very different RAG behaviour:
- **Small chunks (500c)** — more precise retrieval but less context per chunk
- **Medium chunks (1000c)** — balanced, most commonly used in practice
- **Large chunks (2000c)** — more context preserved but retrieval is less targeted

### Why use overlap?
Without overlap, if a relevant sentence falls at a chunk boundary it gets
split across two chunks and may be retrieved incompletely. Overlap ensures
that boundary content appears in **both** the preceding and following chunk,
reducing the chance of losing important context.

### What we store
Results for each configuration are saved in `fixed_results` dictionary
so we can compare all strategies side by side at the end of the lab.

### What to look for in the output
- Are chunk sizes consistent across PDF and podcast?
- Does increasing overlap significantly increase the chunk count?
- How does the character range (min–max) compare between content types?

In [ ]:
fixed_results = {}  # Store results for comparison later

# Experiment with different chunk sizes and overlaps
FIXED_CONFIGS = [
    {'chunk_size': 500,  'chunk_overlap': 0},
    {'chunk_size': 500,  'chunk_overlap': 50},
    {'chunk_size': 1000, 'chunk_overlap': 100},
    {'chunk_size': 2000, 'chunk_overlap': 200},
]

for config in FIXED_CONFIGS:
    size = config['chunk_size']
    overlap = config['chunk_overlap']
    
    # separator='' means split purely on character count (truly fixed-size)
    splitter = CharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=overlap,
        separator='',  # No separator = purely fixed-size
        length_function=len
    )
    
    pdf_chunks = splitter.split_text(pdf_text)
    podcast_chunks = splitter.split_text(podcast_text)
    
    key = f'fixed_{size}_{overlap}'
    fixed_results[key] = {
        'config': config,
        'pdf': chunk_stats(pdf_chunks, f'📄 PDF | Fixed {size} chars, {overlap} overlap'),
        'podcast': chunk_stats(podcast_chunks, f'🎙️ Podcast | Fixed {size} chars, {overlap} overlap'),
        'pdf_chunks': pdf_chunks,
        'podcast_chunks': podcast_chunks
    }

## 🔍 Step 6b: Inspecting Fixed-Size Chunk Boundaries

### What this code does
After generating all fixed-size chunks, we **visually inspect** the first 3
chunks from both the PDF and the podcast using the 1000-char, 100-overlap
configuration — the most commonly used size in practice.

### Why visual inspection matters
Numbers alone (avg tokens, chunk count) don't tell the full story. By reading
the actual chunk content we can answer questions that statistics cannot:

- Does the chunk start mid-word or mid-sentence?
- Does it end at a natural boundary or cut through an idea?
- Is the overlap actually providing useful context continuity?
- Is noisy content (headers, page numbers, dotted lines) being included?

### What to look for

**Signs of poor chunking:**
- A chunk starts with a lowercase letter or partial word (e.g. `y does NOT`)
- A chunk ends abruptly mid-sentence with no punctuation
- Repeated boilerplate content (page headers, version numbers) appears in multiple chunks
- Table of contents entries and dotted separators are treated as content

**Signs of good chunking:**
- Chunks start at the beginning of a sentence or paragraph
- Chunks end with terminal punctuation (`.`, `?`, `!`)
- The content within each chunk is self-contained and meaningful
- Overlap between chunks provides smooth context continuity

### What we found
Fixed-size chunking on the **PDF** included table of contents noise and cut
through words at boundaries (e.g. chunk 2 starting with `y does NOT`).
On the **podcast**, cuts were mid-sentence but the conversational flow
was still somewhat readable due to the continuous nature of the transcript.

In [ ]:
# Inspect sample chunks from the 1000-char, 100-overlap config
key = 'fixed_1000_100'
print('=== PDF Sample Chunks (Fixed 1000 chars) ===\n')
for i, chunk in enumerate(fixed_results[key]['pdf_chunks'][:3]):
    print(f'--- Chunk {i+1} ({len(chunk)} chars) ---')
    print(chunk)
    print()

print('\n=== Podcast Sample Chunks (Fixed 1000 chars) ===\n')
for i, chunk in enumerate(fixed_results[key]['podcast_chunks'][:3]):
    print(f'--- Chunk {i+1} ({len(chunk)} chars) ---')
    print(chunk)
    print()

## 📊 Step 6c: Measuring Sentence Break Quality

### What this code does
We now **quantify** what we observed visually in the previous step.
The `check_sentence_breaks()` function scans the first 10 chunks and
measures what percentage end without terminal punctuation — a signal
that the chunk cut through a sentence rather than ending at a natural boundary.

### Why this metric matters for RAG
When a chunk ends mid-sentence, two problems occur:

1. **The current chunk** ends with an incomplete thought, which weakens
   the embedding quality — the vector may not accurately represent the
   full meaning of the content
2. **The next chunk** starts mid-sentence, meaning it lacks the beginning
   of the thought it contains — if retrieved in isolation, it will be
   confusing to the LLM trying to answer a question from it

### Results from our data

| Content | Mid-sentence breaks |
|---|---|
| PDF | 30% |
| Podcast | 80% |

### What these numbers tell us

**PDF at 30%** — lower than expected for fixed-size chunking, but not
because the strategy is smart. The PDF contains many short lines, headers,
and whitespace that accidentally align with the 1000-char boundary,
creating the illusion of clean cuts. It is coincidental, not structural.

**Podcast at 80%** — nearly every chunk cuts mid-sentence. This is
expected: the Whisper transcript is one continuous block of speech with
no paragraph breaks or structural markers for the splitter to align with.

### Key takeaway
> Fixed-size chunking is fast and simple, but it is blind to meaning.
> It works best when content is highly uniform and sentence length is
> consistent. For structured documents or conversational transcripts,
> a smarter strategy is needed — which is exactly what we explore next
> with Recursive Character Chunking.

In [ ]:
# Analyze sentence break quality
key = 'fixed_1000_100'
pdf_break_pct = check_sentence_breaks(fixed_results[key]['pdf_chunks'])
podcast_break_pct = check_sentence_breaks(fixed_results[key]['podcast_chunks'])

print(f'📄 PDF — {pdf_break_pct:.0f}% of chunks end mid-sentence')
print(f'🎙️ Podcast — {podcast_break_pct:.0f}% of chunks end mid-sentence')
print()

---
## 🔁 Step 7: Strategy 2 — Recursive Character Chunking

### What is Recursive Character Chunking?
Recursive Character Chunking is LangChain's **recommended default strategy**.
Instead of cutting at a fixed character count like the previous approach, it
tries a list of separators in priority order — only falling back to the next
separator if the current one produces chunks that are still too large.

### How the separator priority works
We define this separator list:
```python
separators=['\n\n', '\n', '. ', ' ', '']
```

The splitter works through them in order:

| Priority | Separator | Meaning |
|---|---|---|
| 1st | `'\n\n'` | Split on paragraph breaks (best option) |
| 2nd | `'\n'` | Split on line breaks |
| 3rd | `'. '` | Split on sentence endings |
| 4th | `' '` | Split on word boundaries |
| 5th | `''` | Split on any character (last resort) |

This means it will **always try to keep paragraphs together first**, only
breaking at sentence level if a paragraph is too long, and only breaking
mid-word as an absolute last resort.

### Key difference from Fixed-Size
Fixed-size chunking asks: *"Where is the Nth character?"*  
Recursive chunking asks: *"Where is the nearest natural boundary before N characters?"*

### Why we increased the overlap
Notice the overlap values are larger than in fixed-size (200 vs 100).
This is intentional — because recursive chunks vary more in size, a larger
overlap ensures consistent context continuity at boundaries regardless of
where the split happened to fall.

### What to look for in the output
- Are chunk sizes more variable than fixed-size? (they should be)
- Does the PDF show lower mid-sentence break rates?
- Does the podcast improve — or is the lack of paragraph breaks still a problem?

In [ ]:
recursive_results = {}

RECURSIVE_CONFIGS = [
    {'chunk_size': 500,  'chunk_overlap': 50},
    {'chunk_size': 1000, 'chunk_overlap': 200},
    {'chunk_size': 2000, 'chunk_overlap': 200},
]

for config in RECURSIVE_CONFIGS:
    size = config['chunk_size']
    overlap = config['chunk_overlap']
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=overlap,
        length_function=len,
        # Priority order: try to split on paragraphs, then sentences, then words
        separators=['\n\n', '\n', '. ', ' ', '']
    )
    
    pdf_chunks = splitter.split_text(pdf_text)
    podcast_chunks = splitter.split_text(podcast_text)
    
    key = f'recursive_{size}_{overlap}'
    recursive_results[key] = {
        'config': config,
        'pdf': chunk_stats(pdf_chunks, f'📄 PDF | Recursive {size} chars, {overlap} overlap'),
        'podcast': chunk_stats(podcast_chunks, f'🎙️ Podcast | Recursive {size} chars, {overlap} overlap'),
        'pdf_chunks': pdf_chunks,
        'podcast_chunks': podcast_chunks
    }

## 🔍 Step 7b: Inspecting Recursive Chunk Boundaries

### What this code does
Same visual inspection as we did for fixed-size, but now applied to the
recursive 1000-char, 200-overlap configuration. We read the first 3 chunks
from both the PDF and the podcast to see whether the smarter separator
priority actually produces cleaner boundaries.

### What we found

**PDF — significant improvement:**
- Chunk 2 now starts with `"Please note that..."` — a complete sentence
- Compare this to fixed-size where chunk 2 started with `"y does NOT"` —
  a cut through the middle of the word "automatically"
- The splitter successfully found `'\n\n'` paragraph breaks and used them
  as split points instead of cutting at an arbitrary character count

**Podcast — limited improvement:**
- Chunks still start with `". "` — a leading period from the previous sentence
- This happens because the transcript has **no paragraph breaks at all**
- The splitter had to fall back to `'. '` (sentence endings) as its best
  available separator, which means the period gets consumed and appears
  at the start of the next chunk
- This is not a bug — it is a fundamental limitation of the content type:
  Whisper produces one continuous block of text with no structural markers

### Key insight
> The effectiveness of Recursive Character Chunking is **directly proportional
> to how much structure exists in the source content**. The PDF benefited
> greatly because it had paragraph breaks to split on. The podcast saw minimal
> improvement because it had none — revealing that content type matters more
> than strategy choice when structure is absent.

In [ ]:
# Inspect recursive chunks — notice how they respect sentence/paragraph boundaries
key = 'recursive_1000_200'
print('=== PDF Sample Chunks (Recursive 1000 chars) ===\n')
for i, chunk in enumerate(recursive_results[key]['pdf_chunks'][:3]):
    print(f'--- Chunk {i+1} ({len(chunk)} chars) ---')
    print(chunk)
    print()

print('\n=== Podcast Sample Chunks (Recursive 1000 chars) ===\n')
for i, chunk in enumerate(recursive_results[key]['podcast_chunks'][:3]):
    print(f'--- Chunk {i+1} ({len(chunk)} chars) ---')
    print(chunk)
    print()

## 📊 Step 7c: Comparing Sentence Break Rates — Fixed vs Recursive

### What this code does
We run our `check_sentence_breaks()` function on both strategies side by side
to get a **quantitative comparison** of how often each one cuts mid-sentence.
This turns our visual observations from the previous steps into concrete numbers.

### Issue we encountered — and fixed
When we first ran this comparison, the results were counterintuitive:

| Strategy | PDF | Podcast |
|---|---|---|
| Fixed-size | 30% | 80% |
| Recursive | 80% | 90% |

Recursive appeared **worse** than fixed-size — which made no sense given
what we saw in the visual inspection. The problem was in our measurement
function, not the chunking strategy.

`RecursiveCharacterTextSplitter` splits on `'. '` and **consumes the period**,
meaning chunks end with a word rather than a `.` — our function incorrectly
counted these as mid-sentence breaks even when the split was perfectly clean.

### The fix — `check_sentence_breaks_v2()`
We rewrote the function to detect **true mid-word cuts** instead of checking
for terminal punctuation:
```python
# A real break = chunk ends with a lowercase letter (cut mid-word)
is_mid_word = last_char.isalpha() and last_char.islower()
```

### Corrected results

| Strategy | PDF | Podcast |
|---|---|---|
| Fixed-size | 10% | 80% |
| Recursive | 0% | 80% |

### What this tells us
- **PDF:** Recursive achieves 0% mid-sentence breaks — a clear win over fixed-size
- **Podcast:** Both strategies sit at 80% — confirming that the transcript's
  lack of structure is the limiting factor, not the strategy itself

> 💡 **Lesson learned:** Always validate your measurement functions against
> known outputs before drawing conclusions. A bug in the metric can make a
> better strategy appear worse than an inferior one.

In [ ]:
# Compare sentence break rates between fixed and recursive
key_r = 'recursive_1000_200'
key_f = 'fixed_1000_100'

print('📊 Sentence Break Comparison (1000-char chunks)\n')
print(f'{"Strategy":<20} {"PDF":<20} {"Podcast":<20}')
print('-' * 60)

for label, key in [('Fixed-size', key_f), ('Recursive', key_r)]:
    results = fixed_results if 'fixed' in key else recursive_results
    pdf_pct = check_sentence_breaks(results[key]['pdf_chunks'])
    pod_pct = check_sentence_breaks(results[key]['podcast_chunks'])
    print(f'{label:<20} {pdf_pct:.0f}% mid-sentence    {pod_pct:.0f}% mid-sentence')

## 🔧 Step 7d: Improved Sentence Break Measurement — `check_sentence_breaks_v2()`

### Why we needed a new function
The original `check_sentence_breaks()` flagged any chunk not ending with
terminal punctuation (`.`, `!`, `?`) as a mid-sentence break. This worked
fine for fixed-size chunking, but produced misleading results for recursive
chunking because of how `RecursiveCharacterTextSplitter` handles its separators.

When splitting on `'. '`, the splitter **consumes the period and the space**,
meaning the preceding chunk ends with a word — not a period. Our original
function incorrectly counted this as a break even when the split was clean.

### What changed in v2
The new function detects only **true mid-word cuts** — cases where a chunk
ends with a lowercase letter, indicating the splitter broke through a word:
```python
# Original logic — too strict, flags clean recursive cuts
if stripped[-1] not in '.!?"\'':
    breaks += 1

# Improved logic — only flags genuine mid-word breaks
is_mid_word = last_char.isalpha() and last_char.islower()
if is_mid_word:
    breaks += 1
```

### Why this distinction matters

| Ending | Example | Verdict |
|---|---|---|
| Ends with `.` | `"...of the AI Act."` | ✅ Clean cut |
| Ends with word (period consumed) | `"...of the AI Act"` | ✅ Clean cut |
| Ends mid-word | `"...automaticall"` | ❌ True break |

### Corrected results after the fix

| Strategy | PDF | Podcast |
|---|---|---|
| Fixed-size | 10% | 80% |
| Recursive | 0% | 80% |

These numbers now correctly reflect what we observed visually — recursive
chunking achieves perfect sentence boundary preservation on the PDF,
while both strategies struggle equally with the structureless podcast transcript.

In [ ]:
def check_sentence_breaks_v2(chunks: list, sample_size: int = 10) -> float:
    """
    Improved version: also accepts chunks ending with a word
    that was cleanly cut at a sentence boundary (period consumed by splitter).
    """
    sample = chunks[:sample_size]
    breaks = 0
    
    for chunk in sample[:-1]:
        stripped = chunk.strip()
        if not stripped:
            continue
        last_char = stripped[-1]
        # Also peek: does the next chunk start with a capital? (clean cut)
        is_clean_ending = last_char in '.!?"\')'
        is_mid_word = last_char.isalpha() and last_char.islower()
        if is_mid_word:
            breaks += 1
    
    return (breaks / len(sample)) * 100 if sample else 0

# Re-run comparison with fixed function
print(f'📊 Fixed vs Recursive (corrected measurement)\n')
print(f'{"Strategy":<20} {"PDF":<20} {"Podcast"}')
print('-' * 60)

pdf_fixed_break  = check_sentence_breaks_v2(fixed_results['fixed_1000_100']['pdf_chunks'])
pod_fixed_break  = check_sentence_breaks_v2(fixed_results['fixed_1000_100']['podcast_chunks'])
pdf_recur_break  = check_sentence_breaks_v2(recursive_results['recursive_1000_200']['pdf_chunks'])
pod_recur_break  = check_sentence_breaks_v2(recursive_results['recursive_1000_200']['podcast_chunks'])

print(f'{"Fixed-size":<20} {pdf_fixed_break:.0f}% mid-sentence    {pod_fixed_break:.0f}% mid-sentence')
print(f'{"Recursive":<20} {pdf_recur_break:.0f}% mid-sentence    {pod_recur_break:.0f}% mid-sentence')

---
## ## 🔢 Step 8: Strategy 3 — Token-Based Chunking

### What is Token-Based Chunking?
Token-based chunking splits text based on **token count** rather than
character count. This is the most accurate strategy for production RAG
systems because LLMs measure their context window in tokens — not characters.

### Why tokens and not characters?
One character ≠ one token. The relationship between characters and tokens
varies significantly depending on the content:

| Content type | Chars per token (approx) |
|---|---|
| Common English words | ~4 chars |
| Whitespace / punctuation | ~1 char |
| Long or technical words | ~6-8 chars |
| Dotted lines (`......`) | ~1 char |

This means a 1000-character chunk from the podcast might use 207 tokens,
while a 1000-character chunk from the PDF (with lots of dots and whitespace)
might use only 192 tokens. Character-based chunking cannot account for this
variation — token-based chunking guarantees it.

### How it works in code
We use LangChain's `TokenTextSplitter` with `encoding_name='cl100k_base'` —
the same tokenizer used by GPT-4. This ensures our chunk sizes are directly
comparable to the model's actual context window limits.

### Configurations we test

| chunk_size | chunk_overlap | Typical use case |
|---|---|---|
| 256 tokens | 25 | Short, precise retrieval |
| 512 tokens | 50 | Balanced — most common in production |
| 1024 tokens | 100 | Long context, complex reasoning tasks |

### What to look for in the output
- Token ranges should be extremely tight (close to the requested size)
- Character counts will vary widely — this is expected and correct
- Fewer total chunks than recursive because token chunks are larger
- Compare chunk counts for PDF vs podcast — the short podcast will
  produce very few chunks at larger token sizes

In [ ]:
token_results = {}

TOKEN_CONFIGS = [
    {'chunk_size': 256,  'chunk_overlap': 25},
    {'chunk_size': 512,  'chunk_overlap': 50},
    {'chunk_size': 1024, 'chunk_overlap': 100},
]

for config in TOKEN_CONFIGS:
    size = config['chunk_size']
    overlap = config['chunk_overlap']
    
    splitter = TokenTextSplitter(
        chunk_size=size,
        chunk_overlap=overlap,
        encoding_name='cl100k_base'  # Same encoder as GPT-4
    )
    
    pdf_chunks = splitter.split_text(pdf_text)
    podcast_chunks = splitter.split_text(podcast_text)
    
    key = f'token_{size}_{overlap}'
    token_results[key] = {
        'config': config,
        'pdf': chunk_stats(pdf_chunks, f'📄 PDF | Token {size} tokens, {overlap} overlap'),
        'podcast': chunk_stats(podcast_chunks, f'🎙️ Podcast | Token {size} tokens, {overlap} overlap'),
        'pdf_chunks': pdf_chunks,
        'podcast_chunks': podcast_chunks
    }

## 🔍 Step 8b: Verifying Token Count Accuracy

### What this code does
We verify that the `TokenTextSplitter` actually produced chunks of the
requested size by comparing the **requested token count** against the
**actual token count** measured independently with `tiktoken`.

### Why verify?
It is good practice in any data pipeline to validate outputs against
expectations. If the splitter were using a different tokenizer internally
than we assumed, the actual counts could differ significantly from what
we requested — which would break context window compliance in production.

### What we found

| Chunk | Requested | Actual Tokens | Chars |
|---|---|---|---|
| 1 | 512 | 512 | 2,915 |
| 2 | 512 | 512 | 4,415 |
| 3 | 512 | 512 | 4,455 |
| 4 | 512 | 512 | 3,867 |
| 5 | 512 | 512 | 2,684 |

### Key observations

**Token counts are exact** — every chunk hits precisely 512 tokens,
confirming that `TokenTextSplitter` and our `count_tokens()` function
are using the same `cl100k_base` tokenizer consistently.

**Character counts vary wildly** — chunks 2 and 3 have nearly identical
token counts (512) but very different character counts (4,415 vs 4,455).
This happens because different sections of the PDF contain different
densities of whitespace, dotted lines, and special characters — all of
which consume tokens at a different rate than regular words.

### Why this matters for production RAG
> If you sized your chunks by characters and assumed 4 chars per token,
> you would be wrong for up to 50% of your chunks. Token-based chunking
> is the **only reliable way** to guarantee that every chunk fits within
> your LLM's context window — regardless of content type or formatting.

In [ ]:
# Verify actual token counts vs requested
key = 'token_512_50'
print('🔍 Token verification for 512-token chunks:\n')
print(f'{"Chunk":<8} {"Requested":<12} {"Actual Tokens":<16} {"Chars"}')
print('-' * 50)

for i, chunk in enumerate(token_results[key]['pdf_chunks'][:5]):
    actual_tokens = count_tokens(chunk)
    print(f'{i+1:<8} {512:<12} {actual_tokens:<16} {len(chunk)}')

---
## ## 🧠 Step 9: Strategy 4 — Semantic Chunking (Advanced)

### What is Semantic Chunking?
Semantic chunking is the most sophisticated strategy we explore. Instead of
splitting based on character count or token count, it splits based on
**meaning** — starting a new chunk whenever the topic or idea shifts
significantly between adjacent sentences.

### How it works
The function follows four steps:

**1. Sentence splitting**  
The text is split into individual sentences on `'. '` (period + space),
giving us the smallest meaningful unit to work with.

**2. Embedding**  
Each sentence is converted into a **vector embedding** using
`all-MiniLM-L6-v2` — a lightweight but capable sentence transformer model.
The embedding captures the semantic meaning of each sentence as a point
in 384-dimensional space.

**3. Similarity comparison**  
For each pair of adjacent sentences, we compute **cosine similarity** —
a score from 0 to 1 measuring how related their meanings are:
- Score close to **1.0** = sentences are about the same topic → keep together
- Score close to **0.0** = sentences shift to a different topic → start new chunk

**4. Chunk boundary decision**  
A new chunk starts when either condition is met:
- Cosine similarity drops below the `threshold` parameter
- The current chunk would exceed `max_chunk_chars` (safety cap)

### The threshold parameter
The threshold is the most important tuning parameter in semantic chunking:

| Threshold | Behaviour |
|---|---|
| 0.75 (high) | Splits very aggressively — almost every sentence becomes its own chunk |
| 0.30 (medium) | Splits on moderate topic shifts |
| 0.05 (low) | Only splits on major topic changes — produces large chunks |

### Issues we encountered
Finding the right threshold required multiple iterations:
- **0.75** → 70-100 tiny chunks averaging ~50-90 chars (too aggressive)
- **0.30** → still too many small chunks
- **0.15** → better but PDF noise still causing problems
- **0.05** → produced 7 PDF chunks and 12 podcast chunks at reasonable sizes

We also discovered that the PDF's **table of contents dotted lines**
(`"..............................."`) scored similarity=1.0 with each other
then dropped to ~0.03 when real content resumed — confusing the splitter.
This required a `clean_text_for_semantic()` preprocessing step to remove
noisy lines before encoding.

### Why run on a sample only?
Encoding 100+ sentences on CPU takes 20-40 seconds. For a full 73-page
document that would mean thousands of sentences and several minutes of
processing. In production, semantic chunking would run on a GPU or via
an API-based embedding service — not a local CPU.

In [ ]:
from sentence_transformers import SentenceTransformer

# Load a lightweight embedding model
print('Loading embedding model (first time may download ~80MB)...')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print('✅ Model loaded')

def semantic_chunk(text: str, threshold: float = 0.75, max_chunk_chars: int = 2000) -> list:
    """
    Split text based on semantic similarity between adjacent sentences.
    
    A new chunk starts when:
    - Cosine similarity between consecutive sentences drops below `threshold`
    - OR the current chunk would exceed `max_chunk_chars`
    
    Args:
        text: Input text to chunk
        threshold: Similarity threshold (0-1). Lower = more aggressive splitting.
        max_chunk_chars: Safety cap on chunk size
    
    Returns:
        List of text chunks
    """
    # Split into sentences
    sentences = [s.strip() for s in text.replace('\n', ' ').split('. ') if s.strip()]
    
    if len(sentences) < 2:
        return [text]
    
    print(f'  Encoding {len(sentences)} sentences...')
    embeddings = embed_model.encode(sentences, show_progress_bar=False)
    
    # Normalize embeddings for cosine similarity
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    embeddings_norm = embeddings / (norms + 1e-10)
    
    chunks = []
    current_chunk = [sentences[0]]
    current_len = len(sentences[0])
    
    for i in range(1, len(sentences)):
        # Cosine similarity between adjacent sentence embeddings
        similarity = float(np.dot(embeddings_norm[i-1], embeddings_norm[i]))
        sentence_len = len(sentences[i])
        
        # Split if: low similarity OR chunk would get too large
        if similarity < threshold or (current_len + sentence_len) > max_chunk_chars:
            chunks.append('. '.join(current_chunk) + '.')
            current_chunk = [sentences[i]]
            current_len = sentence_len
        else:
            current_chunk.append(sentences[i])
            current_len += sentence_len
    
    # Don't forget the last chunk
    if current_chunk:
        chunks.append('. '.join(current_chunk) + '.')
    
    return chunks

print('✅ Semantic chunking function defined')

## 🧪 Step 9b: Running Semantic Chunking on a Sample

### What this code does
We apply the `semantic_chunk()` function to the **first 5000 characters**
of both the PDF and the podcast, then compute statistics to compare against
the other strategies.

### Why 5000 characters and not the full text?
The full PDF has 226,399 characters — encoding all of its sentences would
take several minutes on CPU. A 5000-character sample gives us enough content
to observe how semantic chunking behaves without the computational cost.
This is a common and accepted approach when the goal is comparison rather
than production deployment.

### Why threshold=0.3?
This was the result of several iterations of trial and error:

| Attempt | Threshold | Result | Problem |
|---|---|---|---|
| 1st | 0.75 | 70-100 chunks, avg ~50 chars | Way too aggressive |
| 2nd | 0.30 | 55-81 chunks, avg ~60-90 chars | Still too small |
| 3rd | 0.15 | 26-44 chunks, avg ~112-190 chars | Getting better |
| 4th | 0.05 | 7-12 chunks, avg ~415-709 chars | Most usable |

We settled on showing threshold=0.3 here as an intermediate step to
illustrate the tuning process. The final usable results came at 0.05
after also adding text cleaning to remove PDF noise.

### What to look for in the output
- Are chunk sizes more variable than token-based or recursive?
- Does the PDF produce fewer or more chunks than the podcast?
- Does the average token count suggest chunks are too small or too large?

### Important caveat
The numbers from semantic chunking are **not directly comparable** to the
other strategies because they run on a 5000-char sample, while fixed-size,
recursive, and token-based ran on the full text. Semantic chunking is included
here to demonstrate the approach and its trade-offs — not to declare a winner
on raw chunk count alone.

In [ ]:
# Apply to samples (full text would be slow)
SAMPLE_SIZE = 5000

print('\n📄 Semantic chunking PDF sample...')
pdf_chunks_semantic = semantic_chunk(pdf_text[:SAMPLE_SIZE], threshold=0.3)

print('\n🎙️ Semantic chunking Podcast sample...')
podcast_chunks_semantic = semantic_chunk(podcast_text[:SAMPLE_SIZE], threshold=0.3)

semantic_results = {
    'pdf': chunk_stats(pdf_chunks_semantic, '📄 PDF | Semantic (threshold=0.3)'),
    'podcast': chunk_stats(podcast_chunks_semantic, '🎙️ Podcast | Semantic (threshold=0.3)'),
    'pdf_chunks': pdf_chunks_semantic,
    'podcast_chunks': podcast_chunks_semantic
}

## 🔬 Step 9c: Diagnosing the Similarity Score Distribution

### What this code does
Before tuning the threshold, we run a **diagnostic** to understand what
cosine similarity scores actually look like between adjacent sentences in
our specific content. This is essential — without knowing the score range,
picking a threshold is just guessing.

### Why we needed this diagnostic
After our first attempts at semantic chunking produced 70-100 tiny chunks,
we needed to understand why. Rather than keep guessing thresholds, we
inspected the raw similarity scores directly to see what the model was
actually computing.

### What we found in our data
```
Min  : 0.032
Max  : 1.000
Mean : 0.412
```

The scores revealed two critical problems:

**Problem 1 — Dotted lines scoring 1.0**
```
similarity=1.000 | "..............................."
similarity=0.994 | "..........................."
```
The table of contents separator lines are nearly identical strings, so
the model correctly assigns them similarity=1.0. But then when real content
resumes, the score drops to ~0.03 — creating an artificial cliff that
triggers a split even at very low thresholds.

**Problem 2 — Genuine content scoring very low**
```
similarity=0.032 | "Fully implemented practices..."
similarity=0.121 | "1  I..."
```
Short header lines and page markers score extremely low similarity with
surrounding content — further confusing the splitter.

### How this informed our solution
The diagnostic told us two things:
1. No threshold value alone could fix the problem — the noise was too extreme
2. We needed to **clean the text first** before semantic chunking could work

This led us to build the `clean_text_for_semantic()` preprocessing function
that strips dotted lines and short page markers before embedding — removing
the noise that was corrupting the similarity scores.

> 💡 **Lesson learned:** Always inspect your similarity score distribution
> before picking a threshold. What works for clean narrative text will
> completely fail on noisy structured documents like PDFs with tables of
> contents.

In [ ]:
# Diagnostic: see what similarity scores look like between sentences
sentences = [s.strip() for s in pdf_text[:5000].replace('\n', ' ').split('. ') if s.strip()]
embeddings = embed_model.encode(sentences, show_progress_bar=False)
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings_norm = embeddings / (norms + 1e-10)

similarities = []
for i in range(1, min(20, len(sentences))):
    sim = float(np.dot(embeddings_norm[i-1], embeddings_norm[i]))
    similarities.append(sim)
    print(f'Sentences {i}&{i+1}: similarity={sim:.3f} | "{sentences[i][:60]}"')

print(f'\nMin  : {min(similarities):.3f}')
print(f'Max  : {max(similarities):.3f}')
print(f'Mean : {np.mean(similarities):.3f}')

## 🧹 Step 9d: Cleaning PDF Text Before Semantic Chunking

### What this code does
Based on the diagnostic from the previous step, we add a **preprocessing
function** that strips noisy lines from the PDF before encoding. We then
re-run semantic chunking on the cleaned text with a lower threshold of 0.15.

### Why cleaning is necessary for PDFs
The diagnostic revealed that dotted separator lines from the table of contents
were scoring similarity=1.0 with each other, then dropping to ~0.03 when real
content resumed. This created artificial topic-shift signals that no threshold
value could reliably filter out.

The solution is to remove the noise **before** embedding — so the model only
sees meaningful sentences.

### What `clean_text_for_semantic()` removes

| Pattern | Regex | Example |
|---|---|---|
| Dotted separator lines | `^[\s\.]{5,}$` | `"..............................."` |
| Standalone page numbers | `^\d+\s*$` | `"2"`, `"15"` |

### Why the podcast doesn't need cleaning
The Whisper transcript is clean continuous speech — no tables of contents,
no page numbers, no dotted lines. We assign it directly to `podcast_clean`
without any preprocessing.

### Threshold progression so far

| Step | Threshold | PDF Chunks | Avg Chars | Problem |
|---|---|---|---|---|
| 9b | 0.75 | 70 | 70 | Too aggressive |
| 9b | 0.30 | 55 | 89 | Still too small |
| 9d | 0.15 | 26 | 190 | Improving |
| 9e | 0.05 | 7 | 709 | Most usable ✅ |

### Key insight
> Semantic chunking has a hidden dependency that the other strategies do not:
> **input quality**. Fixed-size, recursive, and token-based chunking are
> largely indifferent to noise in the text. Semantic chunking is highly
> sensitive to it — garbage sentences produce garbage embeddings, which
> produce garbage similarity scores, which produce garbage chunks.
> Always clean your text before applying semantic chunking to real-world PDFs.

In [ ]:
import re

def clean_text_for_semantic(text: str) -> str:
    """Remove noisy lines like dotted separators and page headers before semantic chunking."""
    lines = text.split('\n')
    cleaned = []
    for line in lines:
        # Skip lines that are mostly dots or whitespace
        if re.match(r'^[\s\.]{5,}$', line):
            continue
        # Skip short page header lines like "2  Living Repository..."
        if re.match(r'^\d+\s*$', line.strip()):
            continue
        cleaned.append(line)
    return '\n'.join(cleaned)

pdf_clean = clean_text_for_semantic(pdf_text[:5000])
podcast_clean = podcast_text[:5000]  # Podcast doesn't need cleaning

print('\n📄 Semantic chunking cleaned PDF sample...')
pdf_chunks_semantic = semantic_chunk(pdf_clean, threshold=0.15)

print('\n🎙️ Semantic chunking Podcast sample...')
podcast_chunks_semantic = semantic_chunk(podcast_clean, threshold=0.15)

semantic_results = {
    'pdf': chunk_stats(pdf_chunks_semantic, '📄 PDF | Semantic (threshold=0.15)'),
    'podcast': chunk_stats(podcast_chunks_semantic, '🎙️ Podcast | Semantic (threshold=0.15)'),
    'pdf_chunks': pdf_chunks_semantic,
    'podcast_chunks': podcast_chunks_semantic
}

## ⚙️ Step 9e: Final Threshold Tuning — Finding the Sweet Spot

### What this code does
This is the **final iteration** of our semantic chunking tuning process.
After progressively lowering the threshold through steps 9b, 9c, and 9d,
we arrive at `threshold=0.05` — the value that produces the most usable
chunk sizes for both content types.

### Why threshold=0.05?
At this level, the splitter only creates a new chunk when two adjacent
sentences are almost completely unrelated in meaning. This is a very
conservative threshold that prioritises keeping content together over
splitting it apart.

### Final tuning summary

| Threshold | PDF Chunks | PDF Avg Chars | Podcast Chunks | Podcast Avg Chars |
|---|---|---|---|---|
| 0.75 | 70 | 70 | 100 | 49 |
| 0.30 | 55 | 89 | 81 | 60 |
| 0.15 | 26 | 190 | 44 | 112 |
| **0.05** | **7** | **709** | **12** | **415** |

### What `max_chunk_chars=2000` does
This is a safety cap — regardless of similarity scores, no chunk will
exceed 2000 characters. It prevents the extremely low threshold from
grouping the entire document into one single chunk if all sentences
happen to score above 0.05.

### What we found at this threshold

**PDF** — 7 chunks averaging 709 chars with a wide range (4–2004 chars).
The tiny minimum (4 chars) reveals that some noisy content still slipped
through the cleaning step — an inherent limitation of this PDF's structure.

**Podcast** — 12 chunks averaging 415 chars. The semantic splitter
correctly identified topic shifts, for example separating the bridge
metaphor introduction from the discussion of AI algorithms — exactly
the kind of meaningful boundary that character-based strategies miss.

### Final conclusion on semantic chunking
> Semantic chunking works best on **clean, narrative content** like the
> podcast. For noisy structured documents like this PDF, even after
> cleaning, the irregular structure (headers, short entries, metadata)
> limits its effectiveness. The computational cost and tuning effort
> required make it a poor fit for document repositories — but a strong
> choice for transcripts, articles, and essays.

In [ ]:
print('\n📄 Semantic chunking cleaned PDF sample...')
pdf_chunks_semantic = semantic_chunk(pdf_clean, threshold=0.05, max_chunk_chars=2000)

print('\n🎙️ Semantic chunking Podcast sample...')
podcast_chunks_semantic = semantic_chunk(podcast_clean, threshold=0.05, max_chunk_chars=2000)

semantic_results = {
    'pdf': chunk_stats(pdf_chunks_semantic, '📄 PDF | Semantic (threshold=0.05)'),
    'podcast': chunk_stats(podcast_chunks_semantic, '🎙️ Podcast | Semantic (threshold=0.05)'),
    'pdf_chunks': pdf_chunks_semantic,
    'podcast_chunks': podcast_chunks_semantic
}

In [ ]:
# Inspect semantic chunks
print('=== PDF Semantic Chunks ===\n')
for i, chunk in enumerate(pdf_chunks_semantic[:3]):
    print(f'--- Chunk {i+1} ({len(chunk)} chars) ---')
    print(chunk)
    print()

print('\n=== Podcast Semantic Chunks ===\n')
for i, chunk in enumerate(podcast_chunks_semantic[:3]):
    print(f'--- Chunk {i+1} ({len(chunk)} chars) ---')
    print(chunk)
    print()

---
## 📊 Step 10: Visualizing Chunk Size Distributions

### What this code does
We create a **2×4 grid of histograms** comparing chunk size distributions
across all four strategies for both content types. Each histogram shows
how token lengths are distributed across chunks, with a red dashed line
marking the mean.

### How to read the chart

- **X-axis** — chunk size in tokens
- **Y-axis** — number of chunks at that size
- **Red dashed line** — mean chunk size
- **Top row (blue)** — PDF results
- **Bottom row (orange)** — Podcast results

### What the chart reveals

**Fixed (1000c) — tight, symmetric distribution**  
Chunks cluster around the mean with low variance. Predictable but
semantically blind — the consistency comes from mechanical cutting,
not intelligent boundary detection.

**Recursive (1000c) — wider, left-skewed distribution**  
More variance than fixed-size because the splitter respects natural
boundaries, which vary in length. The left tail shows smaller chunks
created when a paragraph or sentence was shorter than the target size.

**Token (512t) — extremely tight, right-skewed distribution**  
Almost all chunks pile up at exactly 511-512 tokens — the tightest
distribution of all four strategies. This confirms that token-based
chunking provides the most reliable context window compliance.

**Semantic (sample) — wide, irregular distribution**  
The most spread out distribution by far, with chunks ranging from
near-zero to 400 tokens. This reflects the content-driven nature of
the strategy — chunk size is determined by topic length, not a size target.
The many near-zero chunks in the PDF confirm the noise problem we
identified and addressed in earlier steps.

### Why this visualization matters
> Numbers in a table tell you the mean. A histogram tells you the
> **full story** — whether chunks are consistently sized or wildly
> variable, whether there are outliers, and whether the strategy
> behaves differently on different content types. Always visualize
> your chunk distributions before choosing a strategy for production.

### NOTE: see chunk_distributions.png

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Chunk Size Distributions by Strategy', fontsize=14, fontweight='bold')

strategies = {
    'Fixed\n(1000c)': {
        'pdf': fixed_results['fixed_1000_100']['pdf']['token_lengths'],
        'podcast': fixed_results['fixed_1000_100']['podcast']['token_lengths']
    },
    'Recursive\n(1000c)': {
        'pdf': recursive_results['recursive_1000_200']['pdf']['token_lengths'],
        'podcast': recursive_results['recursive_1000_200']['podcast']['token_lengths']
    },
    'Token\n(512t)': {
        'pdf': token_results['token_512_50']['pdf']['token_lengths'],
        'podcast': token_results['token_512_50']['podcast']['token_lengths']
    },
    'Semantic\n(sample)': {
        'pdf': semantic_results['pdf']['token_lengths'],
        'podcast': semantic_results['podcast']['token_lengths']
    }
}

colors_pdf     = '#4C72B0'
colors_podcast = '#DD8452'

for col, (strategy_name, data) in enumerate(strategies.items()):
    for row, (content, color) in enumerate([('pdf', colors_pdf), ('podcast', colors_podcast)]):
        ax = axes[row][col]
        token_lengths = data[content]
        ax.hist(token_lengths, bins=15, color=color, alpha=0.85, edgecolor='white')
        ax.axvline(np.mean(token_lengths), color='red', linestyle='--',
                   linewidth=1.5, label=f'Mean: {int(np.mean(token_lengths))}')
        ax.set_title(f'{"📄 PDF" if row==0 else "🎙️ Podcast"}\n{strategy_name}', fontsize=9)
        ax.set_xlabel('Tokens')
        ax.set_ylabel('Count')
        ax.legend(fontsize=8)
        ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('chunk_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('📁 Saved to chunk_distributions.png')

## 📊 Step 10b: Comparing Total Chunk Counts by Strategy

### What this code does
We create a **grouped bar chart** comparing the total number of chunks
produced by each strategy for the PDF vs the podcast. Unlike the histograms
which showed distribution shape, this chart focuses on a single question:
**how many chunks does each strategy produce?**

### Why chunk count matters for RAG
Chunk count directly affects two things in a RAG system:

1. **Embedding cost** — every chunk gets embedded and stored in the vector
   database. More chunks = higher storage cost and longer indexing time
2. **Retrieval granularity** — more chunks means more precise retrieval
   but also more chances for relevant content to be spread across many chunks

### What the chart shows

| Strategy | PDF Chunks | Podcast Chunks |
|---|---|---|
| Fixed 1000 | 252 | 19 |
| Recursive 1000 | 299 | 22 |
| Token-512 | 94 | 8 |

**Recursive produces the most chunks for the PDF (299)**  
Because it respects paragraph and sentence boundaries, it often creates
chunks smaller than the 1000-char target when a natural boundary appears
before the limit is reached — resulting in more, smaller chunks overall.

**Token-based produces the fewest chunks (94 PDF, 8 Podcast)**  
Because 512 tokens ≈ 2000-4000 characters, each chunk is much larger
than the 1000-char chunks from fixed and recursive strategies. Fewer,
larger chunks trade retrieval precision for context richness.

**Podcast always produces far fewer chunks than PDF**  
The podcast transcript is only 16,691 characters vs 226,399 for the PDF —
roughly 14x shorter. This means token-based chunking at 512 tokens produces
only 8 podcast chunks, which may be too coarse for meaningful retrieval.

### Key takeaway
> There is no universally correct chunk count — it depends on your content
> length, retrieval precision requirements, and embedding budget. Use this
> chart to understand the **scale implications** of each strategy before
> committing to one for production.

### NOTE: See chuck_count_comparison.png

In [ ]:
def plot_chunk_count_comparison():
    """
    Bar chart comparing total number of chunks per strategy for PDF vs Podcast.
    """
    strategies = ['Fixed 1000', 'Recursive 1000', 'Token-512']
    pdf_counts = [
        fixed_results['fixed_1000_100']['pdf']['count'],
        recursive_results['recursive_1000_200']['pdf']['count'],
        token_results['token_512_50']['pdf']['count'],
    ]
    podcast_counts = [
        fixed_results['fixed_1000_100']['podcast']['count'],
        recursive_results['recursive_1000_200']['podcast']['count'],
        token_results['token_512_50']['podcast']['count'],
    ]
    
    x = np.arange(len(strategies))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(9, 5))
    bars1 = ax.bar(x - width/2, pdf_counts,     width, label='PDF',     color='#4C72B0', alpha=0.85)
    bars2 = ax.bar(x + width/2, podcast_counts, width, label='Podcast', color='#DD8452', alpha=0.85)
    
    ax.set_title('Total Chunk Count by Strategy', fontsize=13, fontweight='bold')
    ax.set_xlabel('Chunking Strategy')
    ax.set_ylabel('Number of Chunks')
    ax.set_xticks(x)
    ax.set_xticklabels(strategies)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.savefig('chunk_count_comparison.png', dpi=150)
    plt.show()
    print('📁 Saved chart to chunk_count_comparison.png')

plot_chunk_count_comparison()

## 📋 Step 11: Final Strategy Comparison Summary Table

### What this code does
We bring all results together into a single **formatted summary table**
that shows every strategy, every content type, and the key metrics
side by side in one place. This is the central reference for making
our final recommendations.

### How to read the table

| Column | Description |
|---|---|
| `Strategy` | Chunking approach and configuration used |
| `Content` | PDF or Podcast |
| `Chunks` | Total number of chunks produced |
| `Avg Tokens` | Average chunk size in tokens |
| `Min / Max` | Smallest and largest chunk in tokens |
| `Mid-Sentence%` | Percentage of chunks ending mid-sentence |

### What to focus on

**Chunks column** — reflects retrieval granularity. Too few chunks
(like Token-based Podcast at 8) means retrieval is coarse. Too many
(like Recursive PDF at 299) increases embedding and storage cost.

**Avg Tokens column** — reflects how much context each retrieved chunk
provides to the LLM. A chunk of 89 tokens (Semantic Podcast) gives
very little context. A chunk of 511 tokens (Token PDF) gives rich context.

**Min/Max range** — reflects consistency. A tight range like 430–512
(Token-based) means predictable context window usage. A wide range
like 1–396 (Semantic PDF) means highly unpredictable behaviour.

**Mid-Sentence%** — reflects boundary quality. Lower is better.
Note that this metric uses `check_sentence_breaks_v2()` for accuracy —
see Step 7d for why the original function gave misleading results.

### Important note on Semantic results
The Semantic rows are based on a **5000-character sample only**, while
all other strategies ran on the full text. Semantic chunk counts and
averages are therefore not directly comparable — they are included to
illustrate the strategy's behaviour and trade-offs, not to rank it
on equal footing with the others.

In [ ]:
def print_summary_table():
    """
    Print a formatted comparison table across all strategies.
    """
    print('\n' + '='*85)
    print('📋 STRATEGY COMPARISON SUMMARY')
    print('='*85)
    
    header = f'{"Strategy":<22} {"Content":<10} {"Chunks":<8} {"Avg Tokens":<13} {"Min":<8} {"Max":<8} {"Mid-Sentence%"}'
    print(header)
    print('-'*85)
    
    rows = [
        ('Fixed (1000c, 100ov)', 'PDF',     fixed_results['fixed_1000_100']['pdf'],     fixed_results['fixed_1000_100']['pdf_chunks']),
        ('Fixed (1000c, 100ov)', 'Podcast', fixed_results['fixed_1000_100']['podcast'], fixed_results['fixed_1000_100']['podcast_chunks']),
        ('Recursive (1000c)',    'PDF',     recursive_results['recursive_1000_200']['pdf'],     recursive_results['recursive_1000_200']['pdf_chunks']),
        ('Recursive (1000c)',    'Podcast', recursive_results['recursive_1000_200']['podcast'], recursive_results['recursive_1000_200']['podcast_chunks']),
        ('Token-based (512t)',   'PDF',     token_results['token_512_50']['pdf'],     token_results['token_512_50']['pdf_chunks']),
        ('Token-based (512t)',   'Podcast', token_results['token_512_50']['podcast'], token_results['token_512_50']['podcast_chunks']),
        ('Semantic (sample)',    'PDF',     semantic_results['pdf'],     semantic_results['pdf_chunks']),
        ('Semantic (sample)',    'Podcast', semantic_results['podcast'], semantic_results['podcast_chunks']),
    ]
    
    for strategy, content, stats, chunks in rows:
        break_pct = check_sentence_breaks(chunks)
        print(f'{strategy:<22} {content:<10} {stats["count"]:<8} {stats["avg_tokens"]:<13} '
              f'{stats["min_tokens"]:<8} {stats["max_tokens"]:<8} {break_pct:.0f}%')
    
    print('='*85)

print_summary_table()

---
## 💡 Step 11: Final Analysis & Recommendations

In [ ]:
recommendations = """## Chunking Strategy Recommendations

### For PDF Documents:
**Recommended Strategy:** Token-Based Chunking
**Config:** chunk_size=512 tokens, chunk_overlap=50

**Reasoning:**
- This PDF (73 pages, 226k chars) is a structured repository with headers,
  tables of contents, and dotted separators that confuse semantic splitters
- Token-based gives exactly 94 chunks with a tight 430-512 token range,
  guaranteeing compliance with LLM context windows
- Recursive produced 299 chunks (too granular) due to the many short lines
  and headers in the document
- Semantic chunking struggled with the noisy table of contents structure

**Optimal config:** 512 tokens, 50 overlap (~10% overlap rate)

---

### For Podcast Transcripts:
**Recommended Strategy:** Recursive Character Chunking
**Config:** chunk_size=1000, chunk_overlap=200
**Separators:** ['. ', '? ', '! ', ' ', '']

**Reasoning:**
- Whisper transcripts are one continuous block with no paragraph breaks,
  so paragraph-based splitting offers no advantage
- Recursive with sentence-level separators produces 22 meaningful chunks
  that respect spoken thoughts
- Token-based gives only 8 chunks (too coarse for a 16k char transcript)
- Fixed-size cuts mid-sentence 80% of the time — worst option for speech
- Larger overlap (200 chars) preserves conversational continuity across boundaries

**Optimal config:** 1000 chars, 200 overlap (~20% overlap rate)

---

### Trade-offs Summary:

| Strategy      | Pros                              | Cons                          | Best For                  |
|---------------|-----------------------------------|-------------------------------|---------------------------|
| Fixed-Size    | Simple, fast, predictable         | Cuts mid-sentence often       | Uniform homogeneous text  |
| Recursive     | Respects structure, flexible      | Uneven sizes, more chunks     | Transcripts, prose docs   |
| Token-Based   | Exact LLM context compliance      | Ignores semantic boundaries   | Production RAG pipelines  |
| Semantic      | Groups by meaning                 | Slow, noisy PDFs break it     | Clean narrative content   |

---

### Key Findings:
1. **Content type matters more than strategy choice** — the PDF structural
   noise (dotted lines, headers) was the biggest obstacle across all strategies
2. **Semantic chunking requires clean input** — it correctly split the podcast
   by topic (bridge metaphor vs AI algorithms) but failed on the noisy PDF
3. **Token-based is safest for production** — guaranteed context window compliance
   regardless of content type
4. **Overlap is critical for transcripts** — 20% overlap recommended to preserve
   conversational context across boundaries
"""

print(recommendations)

with open('recommendations.md', 'w', encoding='utf-8') as f:
    f.write(recommendations)
print('📁 Saved to recommendations.md')

# 📝 Lab Report: Different Ways to Chunk Podcast and PDF

## Overview
This lab explored four chunking strategies for building a RAG system using two
content types: a Trustworthy AI podcast (`.m4a` audio) and an AI Literacy PDF
(73 pages). The goal was to compare strategies, understand trade-offs, and make
evidence-based recommendations.

---

## What We Did

### Step 1: Environment Setup
Installed all required libraries: `langchain`, `pypdf`, `tiktoken`,
`sentence-transformers`, `pydub`, and `openai`.

### Step 2: Podcast Transcription
Used OpenAI Whisper API (`whisper-1`) to transcribe the podcast audio file.
The transcript was saved locally to avoid re-transcribing on reruns.
- **Result:** 16,691 characters, 2,868 words

### Step 3: PDF Loading
Extracted text from the PDF using `pypdf`, joining pages with double newlines
to preserve structure.
- **Result:** 226,399 characters, 32,066 words across 73 pages

### Step 4: Fixed-Size Chunking
Split both documents using `CharacterTextSplitter` with configs:
500, 1000, and 2000 characters with varying overlap values.

### Step 5: Recursive Character Chunking
Used `RecursiveCharacterTextSplitter` with separator priority:
`['\n\n', '\n', '. ', ' ', '']` — tries paragraph breaks first,
then sentences, then words.

### Step 6: Token-Based Chunking
Used `TokenTextSplitter` with `cl100k_base` encoding (GPT-4 tokenizer)
at 256, 512, and 1024 token sizes.

### Step 7: Semantic Chunking
Used `sentence-transformers` (`all-MiniLM-L6-v2`) to embed sentences
and split where cosine similarity dropped between adjacent sentences.
Applied to a 5000-character sample due to computational cost.

### Step 8: Visualization & Comparison
Generated chunk size distribution histograms and a chunk count bar chart
across all strategies for both content types.

---

## Issues Faced & How We Solved Them

### Issue 1: Whisper API 25MB File Size Limit
**Error:** `413 - Maximum content size limit (26214400) exceeded`  
**Cause:** The podcast file was 28.8MB, slightly over the API's 25MB limit.  
**Solution:** Used `pydub` to split the audio into 2 equal chunks (~7.2MB each),
transcribed each separately, then joined the transcripts with a space.

### Issue 2: ffmpeg Not Found
**Error:** `JSONDecodeError` when pydub tried to call `ffprobe`  
**Cause:** ffmpeg was installed at `C:\ffmpeg\fullbuild\bin\` but was not in
the system PATH, so pydub couldn't find it.  
**Solution:** Manually overrode pydub's internal path functions:
```python
AudioSegment.converter = r"C:\ffmpeg\fullbuild\bin\ffmpeg.exe"
AudioSegment.ffprobe   = r"C:\ffmpeg\fullbuild\bin\ffprobe.exe"
pydub.utils.get_prober_name = lambda: r"C:\ffmpeg\fullbuild\bin\ffprobe.exe"
pydub.utils.get_encoder_name = lambda: r"C:\ffmpeg\fullbuild\bin\ffmpeg.exe"
```

### Issue 3: Inaccurate Sentence Break Measurement
**Error:** Recursive chunking showed 80-90% mid-sentence breaks — worse than fixed-size  
**Cause:** `RecursiveCharacterTextSplitter` splits on `. ` and *consumes* the period,
so chunks end without terminal punctuation even when the split was clean.  
**Solution:** Rewrote `check_sentence_breaks()` to detect mid-word cuts
(lowercase last character) rather than checking for terminal punctuation.

### Issue 4: Semantic Chunking Threshold Too High
**Error:** 70-100 tiny chunks with average size of ~50-90 chars  
**Cause:** Default threshold of 0.75 was too aggressive — almost every sentence
pair scored below it, causing a split on nearly every sentence.  
**Solution:** Progressively lowered the threshold from 0.75 → 0.3 → 0.15 → 0.05
until chunks reached a meaningful average size (~400-700 chars).

### Issue 5: PDF Noise Breaking Semantic Chunking
**Error:** Semantic chunks were fragmented by table of contents dotted lines  
**Cause:** Strings like `"..............................."` are nearly identical,
scoring similarity=1.0 with each other, then dropping to ~0.03 when real content
resumed — confusing the splitter.  
**Solution:** Added a `clean_text_for_semantic()` preprocessing function that
strips lines matching `^[\s\.]{5,}$` before semantic chunking.

---

## Key Findings

<table>
  <thead>
    <tr>
      <th>Strategy</th>
      <th>PDF Chunks</th>
      <th>Podcast Chunks</th>
      <th>Avg Tokens</th>
      <th>Mid-Sentence</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Fixed (1000c)</td>
      <td>252</td>
      <td>19</td>
      <td>192 / 207</td>
      <td>30% / 80%</td>
    </tr>
    <tr>
      <td>Recursive (1000c)</td>
      <td>299</td>
      <td>22</td>
      <td>164 / 192</td>
      <td>0% / 80%</td>
    </tr>
    <tr>
      <td>Token-based (512t)</td>
      <td>94</td>
      <td>8</td>
      <td>511 / 486</td>
      <td>—</td>
    </tr>
    <tr>
      <td>Semantic (sample)</td>
      <td>7</td>
      <td>12</td>
      <td>105 / 89</td>
      <td>0% / 0%</td>
    </tr>
  </tbody>
</table>

1. **Content type matters more than strategy** — the PDF's structural noise
   (headers, dotted lines, page numbers) was the biggest obstacle across all strategies.
2. **Recursive chunking is best for clean structured text** — achieved 0% mid-sentence
   breaks on the PDF by respecting paragraph boundaries.
3. **Token-based is safest for production** — guaranteed context window compliance
   with tight token ranges (430–512) regardless of content type.
4. **Semantic chunking requires clean input** — correctly identified topic shifts
   in the podcast (bridge metaphor → AI algorithms) but struggled with the noisy PDF.
5. **Podcast transcripts are the hardest to chunk** — Whisper produces one
   continuous block with no paragraph breaks, limiting all structure-based strategies.

---

## Final Recommendations

### For PDF Documents → Token-Based (512 tokens, 50 overlap)
Guarantees LLM context compliance. The PDF's noise makes semantic chunking
unreliable, and recursive produces too many small chunks (299) from short header lines.

### For Podcast Transcripts → Recursive Character (1000 chars, 200 overlap)
Sentence-level separators `['. ', '? ', '! ']` respect spoken thoughts.
20% overlap preserves conversational continuity across boundaries.
Token-based produces too few chunks (8) for a short transcript.